In [1]:
import pandas as pd
from github_helper import from_github

### Load voting session data, voting data and actor data to join together

In [2]:
df_votes = pd.read_csv(from_github("/voting-data/df_votes_all_periods.csv"))
df_voting_sessions = pd.read_csv(from_github("/voting-data/voting_sessions_enriched.csv"))
party_per_period = pd.read_csv(from_github("/actor-data/party_per_period.csv"))

### We do not want to load all of the data from the votes, only what is needed for our calculations. We then join it, first with the votes, and then with the actors and their periods

In [3]:
relevant_topics_for_joining_on_votes = ["afstemning_id", "afstemning_nummer", "afstemning_vedtaget", "Period", "primary_topic", "all_topics", "møde_dato", 'møde_year_month'] #The last one is a homemade one ya know
df_voting_sessions['møde_dato'] = pd.to_datetime(df_voting_sessions['møde_dato'])
df_voting_sessions['møde_year_month'] = df_voting_sessions['møde_dato'].dt.to_period('M')

#Join the voting session with the votes
votes_enriched = df_votes.merge(df_voting_sessions[relevant_topics_for_joining_on_votes], how = "left", on = "afstemning_id")

#Join the votes with the actor information
votes_with_party = votes_enriched.merge(party_per_period, how = "left", on = ["aktørid", "Period"])

#Do a little bit of cleaning
columns_to_drop = ["vote_opdateringsdato"]
df_limited = votes_with_party.drop(columns = columns_to_drop)
df_renamed = df_limited.rename(columns = {"aktør" : "politician"})



#Sanity check
print(df_renamed.groupby("Period")["afstemning_id"].nunique())
print(df_renamed.groupby("Period")["aktørid"].nunique())

Period
65     294
66    1269
67    1778
68    1727
69    2017
70    1688
71    1306
Name: afstemning_id, dtype: int64
Period
65    189
66    216
67    241
68    228
69    237
70    219
71    235
Name: aktørid, dtype: int64


In [4]:
#And save the new data
for period in [65, 66, 67, 68, 69, 70, 71]:
    votes_p = df_renamed[df_renamed["Period"] == period]
    votes_p.to_csv(f"./voting-data/votes_enriched_p{period}.csv", index = False)